In [ ]:
# This chunk of code is calculating breathing distance (distance between SHL0 and DNA end terminal).

In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis import distances
import numpy as np

top = "nucl.pdb"
umbrellas = range(12)  # umbrella window indices
cutoff = 5.0  # not used, kept for consistency
nblocks = 10  # for block averaging

def block_average(values, nblocks=10):
    n = len(values)
    block_size = n // nblocks
    blocks = [np.mean(values[i*block_size:(i+1)*block_size]) for i in range(nblocks)]
    mean = np.mean(blocks)
    stderr = np.std(blocks, ddof=1) / np.sqrt(nblocks)
    return mean, stderr

summary = []

for i in umbrellas:
    traj = f"window{i}.pbc.xtc"
    u = mda.Universe(top, traj)

    SHL0 = u.select_atoms("(chainID A and resid 68:80 or chainID B and resid 68:80) and not name H*")
    dna_end1 = u.select_atoms("(chainID A and resid 136:145 or chainID B and resid 1:10) and not name H*")
    dna_end2 = u.select_atoms("(chainID A and resid 1:10 or chainID B and resid 136:145) and not name H*")

    dist1_list, dist2_list = [], []

    for ts in u.trajectory:
        com_shl0 = SHL0.center_of_mass()
        com_end1 = dna_end1.center_of_mass()
        com_end2 = dna_end2.center_of_mass()

        d1 = distances.distance_array(com_shl0, com_end1)[0][0]
        d2 = distances.distance_array(com_shl0, com_end2)[0][0]

        dist1_list.append(d1)
        dist2_list.append(d2)

    dist1_arr = np.array(dist1_list)
    dist2_arr = np.array(dist2_list)

    # Save per-frame distances
    np.savetxt(f"h3_dna_com_dist_umb{i}.dat",
               np.column_stack((np.arange(1, len(dist1_arr)+1), dist1_arr, dist2_arr)),
               fmt="%d %.3f %.3f",
               header="Frame COM_Dist_End1 COM_Dist_End2", comments="")

    # Block averages
    avg1, err1 = block_average(dist1_arr, nblocks)
    avg2, err2 = block_average(dist2_arr, nblocks)

    with open(f"h3_dna_com_dist_avg_umb{i}.dat", "w") as f:
        f.write("#DistanceType\tAverage(Å)\tError(Å)\n")
        f.write(f"H3_tail–DNA_end1\t{avg1:.3f}\t{err1:.3f}\n")
        f.write(f"H3_tail–DNA_end2\t{avg2:.3f}\t{err2:.3f}\n")

    summary.append([i, avg1, err1, avg2, err2])
    print(f"Completed umbrella {i}")

# Write summary
summary = np.array(summary)
np.savetxt("SHL0_dna_com_dist_summary_all.dat", summary,
           fmt="%d %.3f %.3f %.3f %.3f",
           header="Umbrella Tail1_Avg Tail1_Error Tail2_Avg Tail2_Error",
           comments="")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Load summary ----
data = np.loadtxt("SHL0_dna_com_dist_summary_all.dat", skiprows=1)
columns = ["Umbrella", "Tail1_Avg", "Tail1_Error", "Tail2_Avg", "Tail2_Error"]
df = pd.DataFrame(data, columns=columns)

# ---- Convert Rg from nm → Å ----
df["Umbrella_Ang"] = df["Umbrella"] * 10

# ---- Plot setup ----
fig, ax = plt.subplots(figsize=(10, 6))

# ---- Plot Tail1 ----
ax.errorbar(df["Umbrella_Ang"], df["Tail1_Avg"], yerr=df["Tail1_Error"],
            fmt='o-', color="red", markersize=8, linewidth=2,
            capsize=4, label="SHL0 – DNA End1")

# ---- Plot Tail2 ----
ax.errorbar(df["Umbrella_Ang"], df["Tail2_Avg"], yerr=df["Tail2_Error"],
            fmt='s-', color="blue", markersize=8, linewidth=2,
            capsize=4, label="SHL0 – DNA End2")

# ---- Axis labels ----
ax.set_xlabel("Radius of gyration (Rg) [Å]", fontsize=24, fontweight="bold")
ax.set_ylabel("Distance (Å)", fontsize=24, fontweight="bold")

# ---- Ticks ----
tick_fontsize = 18
ax.tick_params(axis='both', labelsize=tick_fontsize, width=2, length=6, direction='in')

for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontweight('bold')

# ---- Spines ----
for spine in ax.spines.values():
    spine.set_linewidth(2)

# ---- Legend ----
ax.legend(prop={'weight': 'bold', 'size': 16}, loc='upper left')

# ---- Grid ----
ax.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig("SHL0_dna_distance_summary_plot_styled.png", dpi=600)
plt.show()
